#  <font color='#FFE15D'><b>💎 Introduction to RL</b></font>

# 🔴 **Environment**

In [5]:
import gymnasium as gym

# Initialise the environment
env = gym.make("LunarLander-v3", render_mode="human")

# Reset the environment to generate the first observation
observation, info = env.reset(seed=42)
for _ in range(1000):
    # this is where you would insert your policy
    action = env.action_space.sample()

    # step (transition) through the environment with the action
    # receiving the next observation, reward and if the episode has terminated or truncated
    observation, reward, terminated, truncated, info = env.step(action)

    # If the episode has ended then we can reset to start a new episode
    if terminated or truncated:
        observation, info = env.reset()

env.close()

In [10]:
print(f"Action space: {env.action_space}")  
print(f"Sample action: {env.action_space.sample()}")

# Box observation space (continuous values)
print(f"Observation space: {env.observation_space}") 
print(f"Sample observation: {env.observation_space.sample()}") 

Action space: Discrete(4)
Sample action: 3
Observation space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Sample observation: [ 1.3200408  -2.471043    2.3641565   2.1212351   4.345064    0.7342591
  0.24154903  0.9018063 ]


# 🔴 **Agent**

## 🟠 Learning

In [14]:
import gymnasium as gym
from stable_baselines3 import PPO

# Initialize the environment with a custom time limit (handles 'truncated')
env = gym.make("LunarLander-v3", render_mode="human", max_episode_steps=600)

# Define the PPO agent using an MLP (Multi-Layer Perceptron) policy
model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0003)

# Train the agent
print("--- Starting Training Process ---")
model.learn(total_timesteps=10_000)
print("--- Training Completed Successfully! ---")

env.close()

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
--- Starting Training Process ---
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 97.4     |
|    ep_rew_mean     | -196     |
| time/              |          |
|    fps             | 46       |
|    iterations      | 1        |
|    time_elapsed    | 43       |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 97.2        |
|    ep_rew_mean          | -174        |
| time/                   |             |
|    fps                  | 45          |
|    iterations           | 2           |
|    time_elapsed         | 90          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.010744678 |
|    clip_fraction        | 0.0762      |
|    clip_range           |

## 🟠 Inference

In [17]:
# Initialize the environment with a custom time limit (handles 'truncated')
env = gym.make("LunarLander-v3", render_mode="human", max_episode_steps=600)

# Evaluate the trained agent
state, info = env.reset()
for _ in range(1000):
    # Predict the optimal action using the trained policy
    action, _states = model.predict(state, deterministic=True)
    
    state, reward, terminated, truncated, info = env.step(action)
    
    # Reset the environment if the episode naturally ends or is truncated
    if terminated or truncated:
        state, info = env.reset()

env.close()

# 🔴 **REINFORCE**

In [8]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import time

# Define the Policy Network (The red box in the slide)
class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Softmax(dim=-1)
        )
        
    def forward(self, x):
        return self.fc(x)

# Initialize environment and network
env = gym.make('CartPole-v1', render_mode='human')
policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
optimizer = optim.Adam(policy.parameters(), lr=0.01)

# Main training loop (e.g., 1 sample episode)
for episode in range(100):
    state, info = env.reset()
    
    # -------------------------------------------------------------
    # Buffer (Trajectory) Box: Storage for interaction data
    # -------------------------------------------------------------
    saved_log_probs = []
    rewards = []
    
    done = False
    while not done:
        # Interaction Phase
        state_tensor = torch.FloatTensor(state)
        action_probs = policy(state_tensor)
        
        # Sample an action (Stochastic Policy implementation)
        m = Categorical(action_probs)
        action = m.sample()
        
        # Save to Buffer
        saved_log_probs.append(m.log_prob(action))
        
        # Step the environment to get next state and reward
        state, reward, terminated, truncated, info = env.step(action.item())
        rewards.append(reward)
        
        done = terminated or truncated

    # -------------------------------------------------------------
    # Learning Phase: Policy Optimization after episode completion
    # -------------------------------------------------------------
    
    # Calculate total trajectory reward (R_tau)
    R_tau = sum(rewards)
    
    # Calculate Loss (The orange box in the slide)
    policy_loss = -torch.stack(saved_log_probs).sum() * R_tau
    
    # Compute Gradient and Apply Update (The purple and yellow boxes)
    optimizer.zero_grad()
    policy_loss.backward() # Compute gradients
    optimizer.step()        # Update policy network weights
    
    print(f"Episode finished! Total Reward (R_tau): {R_tau}")

    time.sleep(1.5)

env.close()

Episode finished! Total Reward (R_tau): 10.0
Episode finished! Total Reward (R_tau): 15.0
Episode finished! Total Reward (R_tau): 15.0
Episode finished! Total Reward (R_tau): 28.0
Episode finished! Total Reward (R_tau): 16.0
Episode finished! Total Reward (R_tau): 19.0
Episode finished! Total Reward (R_tau): 24.0
Episode finished! Total Reward (R_tau): 27.0
Episode finished! Total Reward (R_tau): 18.0
Episode finished! Total Reward (R_tau): 29.0
Episode finished! Total Reward (R_tau): 29.0
Episode finished! Total Reward (R_tau): 24.0
Episode finished! Total Reward (R_tau): 12.0
Episode finished! Total Reward (R_tau): 38.0
Episode finished! Total Reward (R_tau): 39.0
Episode finished! Total Reward (R_tau): 21.0
Episode finished! Total Reward (R_tau): 16.0
Episode finished! Total Reward (R_tau): 14.0
Episode finished! Total Reward (R_tau): 25.0
Episode finished! Total Reward (R_tau): 10.0
Episode finished! Total Reward (R_tau): 15.0
Episode finished! Total Reward (R_tau): 32.0
Episode fi